### https://www.kaggle.com/competitions/drawing-with-llms

In [2]:
import kagglehub
import pandas as pd


In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = ( None )
load_in_4bit = False 


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print(model.dtype)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA GeForce RTX 4070 Ti SUPER. Max memory: 15.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
torch.bfloat16


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.2.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
df=pd.read_csv('svg_score_train.csv')
df=df[df['score'] > 0.6]
df=df[['topic','svg_code']]

In [6]:
tmp=pd.read_csv('svg_score_train2.csv')
tmp=tmp[tmp['score'] > 0.6]
tmp=tmp[['topic','svg_code']]

In [7]:
df= pd.concat([df, tmp], ignore_index=True)

In [8]:
df.shape

(557, 2)

In [9]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    topics = examples["topic"]  # Using 'topic' as instruction
    svgs = examples["svg_code"]  # Using 'svg_code' as output
    texts = []
    
    for topic, svg_code in zip(topics, svgs):
        # No additional input is needed, so we pass an empty string
        text = alpaca_prompt.format(f"Generate an SVG image for the topic:",topic, svg_code) + EOS_TOKEN
        texts.append(text)
    
    return { "text": texts }

from datasets import Dataset
import pandas as pd
# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Apply formatting function
dataset = dataset.map(formatting_prompts_func, batched=True)


Map:   0%|          | 0/557 [00:00<?, ? examples/s]

In [10]:
# Check dataset sample output
print(dataset["text"][0])

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Generate an SVG image for the topic:

### Input:
'Golden wheat fields under a setting sun',

### Response:
<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">
  <!-- Background for the sky -->
  <rect x="0" y="0" width="200" height="100" fill="orange" opacity="0.7"/>
  
  <!-- Sun -->
  <circle cx="100" cy="50" r="30" fill="yellow" opacity="0.8"/>
  
  <!-- Wheat fields -->
  <rect x="0" y="100" width="200" height="100" fill="goldenrod"/>
  
  <!-- Wheat stalks -->
  <g stroke="saddlebrown" stroke-width="2">
    <line x1="30" y1="100" x2="30" y2="150"/>
    <line x1="50" y1="100" x2="50" y2="150"/>
    <line x1="70" y1="100" x2="70" y2="150"/>
    <line x1="90" y1="100" x2="90" y2="150"/>
    <line x1="110" y1="100" x2="110" y2="150"/>
    <line x1="130" y1="100" x2="130" y2="1

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, 
        max_steps = 350,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 123,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Converting train dataset to ChatML (num_proc=2):   0%|          | 0/557 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/557 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/557 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/557 [00:00<?, ? examples/s]

In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 557 | Num Epochs = 6
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 350
 "-____-"     Number of trainable parameters = 97,255,424


Step,Training Loss
5,0.813100
10,0.666700
15,0.423800
20,0.366600
25,0.299500
30,0.289400
35,0.261100
40,0.208300
45,0.223800
50,0.214000


In [13]:
#This ONLY saves the LoRA adapters, and not the full model.
model.save_pretrained("./lora/lora_model_3b_v1") # Local saving
tokenizer.save_pretrained("./lora/lora_model_3b_v1")

('./lora/lora_model_3b_v1/tokenizer_config.json',
 './lora/lora_model_3b_v1/special_tokens_map.json',
 './lora/lora_model_3b_v1/tokenizer.json')

In [14]:
# import gc
# del model
# gc.collect()
# torch.cuda.empty_cache()

In [15]:
# import psutil

# def kill_large_python_processes():
#     # Loop over all running processes
#     for proc in psutil.process_iter(['pid', 'name', 'memory_info', 'exe']):
#         try:
#             # Check if the process is Python and type is 'C' (for computation)
#             if 'python' in proc.info['name'].lower():
#                 # Check if memory usage is greater than 2048 MB
#                 memory_usage_mb = proc.info['memory_info'].rss / (1024 * 1024)  # Convert bytes to MB
#                 if memory_usage_mb > 2048:
#                     print(f"Killing Python process with PID {proc.info['pid']} using {memory_usage_mb} MB memory")
#                     proc.kill()  # Kill the process
#         except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
#             pass  # Handle processes that might disappear during iteration

# #kill_large_python_processes()

In [16]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
model_name = "./lora/lora_model_3b_v1", # YOUR MODEL YOU USED FOR TRAINING
max_seq_length = 2048,
dtype = (None),
load_in_4bit = False,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference



==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA GeForce RTX 4070 Ti SUPER. Max memory: 15.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [17]:
# alpaca_prompt = You MUST copy from above!
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Please write a SVG code fo rthe given topic?", # instruction
        "Golden sun rising in the east", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 1024, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nPlease write a SVG code fo rthe given topic?\n\n### Input:\nGolden sun rising in the east\n\n### Response:\n<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">\n  <!-- Background representing the sky -->\n  <rect x="0" y="0" width="200" height="200" fill="lightblue" />\n  \n  <!-- Sun representing the dawn -->\n  <circle cx="100" cy="100" r="50" fill="gold" opacity="0.8" />\n  \n  <!-- Sun rays -->\n  <g stroke="gold" stroke-width="2" opacity="0.6">\n    <line x1="100" y1="100" x2="100" y2="50" />\n    <line x1="100" y1="100" x2="140" y2="100" />\n    <line x1="100" y1="100" x2="60" y2="100" />\n    <line x1="100" y1="100" x2="100" y2="150" />\n    <line x1="100" y1="100" x2="60" y2="160" />\n    <line x1="100" y1="100" x2="140" y2="160" />\n  </g>\n</svg>

In [18]:
#save merged 16bit
model.save_pretrained_merged("./lora/lora_16bit_merged_3b_v1", tokenizer, save_method = "merged_16bit",)

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 13.59 out of 31.21 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


  0%|                                                    | 0/28 [00:00<?, ?it/s]
We will save to Disk and not RAM now.
100%|███████████████████████████████████████████| 28/28 [00:05<00:00,  5.07it/s]


Unsloth: Saving tokenizer... Done.
Done.
